In [2]:
from importlib import reload

from prompt_utils import build_prompt
from data_utils import (
    create_submission,
    parse_predictions,
    create_comparison_df,
    read_dataset,
)
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from sqlalchemy.testing import combinations_list
from tqdm import tqdm
from semevalpolar.utils import get_project_root
import pandas as pd
from custom_rules import apply_rules

import os
import ast

In [3]:
batch_size = 10
data_path = os.path.join(
    get_project_root(), "data", "dev_phase", "subtask1", "train", "eng.csv"
)
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [5]:
combined_df = pd.concat(generator_list, ignore_index=True)

In [7]:
predictions = []
usages = []

In [8]:
import time

for batch in tqdm(generator_list):
    response = test_run(
        batch, prompt_path="prompt-ds-main-classifier.txt", model="openai/gpt-5.2"
    )
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 16/16 [01:59<00:00,  7.45s/it]


In [9]:
flat = [x for sub in predictions for x in sub]

In [26]:
combined_df.insert(2, "llm_polarization", pd.Series(flat).reset_index(drop=True))

In [44]:
ground_truths = []

In [45]:
for i in range(len(flat) // batch_size):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [46]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [47]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(
    comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1]
)
cm

array([[ 7,  2],
       [ 1, 10]])

In [48]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

0.85

In [ ]:
correct = comparison[comparison["Predicted"] == comparison["Ground Truth"]]
correct

In [16]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong

,Predicted,Ground Truth,Text
11,1,0,47th District Republicans kicked out of Kent C...
19,1,0,The Demise of Debate? Trumps Refusal To Face H...
21,1,0,War crimes tweeted in real time.
24,1,0,They said voter fraud doesnt exist tho...
33,1,0,Israel is attempting genocide and the U.S. is ...
40,1,0,"CSG holds meeting, students speak on IsraelHam..."
42,1,0,theyre not taking down the Iron dome. ffs people.
43,1,0,Calling for Impeachment. ASAP. impeachmentnow ...
44,1,0,Israel closes border crossing to Gaza workers ...
46,1,0,Donating to the IDF is Lawful Good. I am confu...


In [12]:
submission = create_submission(combined_df, flat)

In [13]:
submission.to_csv(
    os.path.join(get_project_root(), "predictions", "subtask_1", "pred_eng.csv"),
    index=False,
)

In [21]:
comparison.to_csv(
    os.path.join(
        get_project_root(),
        "predictions",
        "dumps",
        "prompting",
        "with-ds-main-classifier-andqwen.csv",
    ),
    index=False,
)